In [ ]:
import os
import pickle
import sys
import threading
import time

import numpy as np
import matplotlib.pyplot as plt

root_dir = os.path.abspath("..")
data_dir = os.path.join(root_dir, "data")
figs_dir = os.path.join(root_dir, "figs", "curr_max_sensitivity_neural_flux_torque")
os.makedirs(data_dir, exist_ok=True)
os.makedirs(figs_dir, exist_ok=True)
if root_dir not in sys.path:
    sys.path.append(root_dir)

from current_setpoints.models.machines import ieee_machine2_trained_neural_flux_and_torque
from current_setpoints.models.forward_model import ForwardModel, Fault, count_peaks_at_limit
from current_setpoints.optimization import ActiveSetOptimizer


def call_with_timeout(timeout, fn, *args, **kwargs):
    """Same mechanism as grid.py's own _call_with_timeout (plain daemon thread, not
    ThreadPoolExecutor -- see that function's docstring for why). Duplicated locally,
    matching every other sweep notebook this session."""
    result = {}

    def _run():
        try:
            result["value"] = fn(*args, **kwargs)
        except Exception as exc:
            result["error"] = exc

    t = threading.Thread(target=_run, daemon=True)
    t.start()
    t.join(timeout)
    if t.is_alive():
        return None, "timeout"
    if "error" in result:
        return None, f"exception: {result['error']!r}"
    return result.get("value"), None

In [ ]:
CURR_MAX = 30.0
VOLT_MAX = 13.0
N_GRID = 30

SPEEDS_RPM = [0.0, 350.0, 700.0, 1050.0, 1400.0, 1750.0]
REDUCTION_FRACS = [0.01, 0.02, 0.05, 0.10, 0.15]
TORQUE_BACKOFF = 0.98

PROBE_TIMEOUT_SECONDS = 180
CELL_TIMEOUT_SECONDS = 180
SLSQP_OPTS = {"disp": False, "ftol": 1e-3, "maxiter": 500}

drive = ieee_machine2_trained_neural_flux_and_torque(curr_max=CURR_MAX, volt_max=VOLT_MAX)
fwd = ForwardModel(drive, n_theta=700, fault=Fault(()))
const_mech_speed = 30.0 / (np.pi * drive.n_ppairs)

optimizer = ActiveSetOptimizer(fwd, opts=SLSQP_OPTS, n_grid=N_GRID)

print(f"{len(SPEEDS_RPM)} speeds x {len(REDUCTION_FRACS)} curr_max levels = "
      f"{len(SPEEDS_RPM) * len(REDUCTION_FRACS)} solves")

In [ ]:
PHASE_LETTERS = ["a", "b", "c", "d", "e"]


def segment_label(n_curr, n_volt):
    roman = {0: "", 1: "I", 2: "II"}
    return f"MTPA_{n_curr}" if n_volt == 0 else f"FW_{roman[n_volt]},{n_curr}"


def fine_idx_grid(fwd, theta_grid):
    n_theta = fwd.vec_theta.size - 1
    return np.round(theta_grid / (2 * np.pi) * n_theta).astype(int) % n_theta


def alpha_beta_xy(theta_grid, sched):
    c1, s1 = np.cos(theta_grid), np.sin(theta_grid)
    c3, s3 = np.cos(3 * theta_grid), np.sin(3 * theta_grid)
    id1, iq1, id3, iq3 = sched[:, 0], sched[:, 1], sched[:, 2], sched[:, 3]
    i_alpha = c1 * id1 - s1 * iq1
    i_beta = s1 * id1 + c1 * iq1
    i_x = c3 * id3 - s3 * iq3
    i_y = s3 * id3 + c3 * iq3
    return i_alpha, i_beta, i_x, i_y


def waveforms_for_plot(fwd, omega, curr_dq_traj):
    """(theta_deg, iph, vph) reconstructed at each node's own angle/current -- ActiveSetOptimizer
    always returns a full (n_grid, dim) trajectory (never a single constant vector), unlike
    StaticOptimizer, so this notebook doesn't need the static branch the map notebook did."""
    kept = list(fwd.fault._kept)
    n_grid = curr_dq_traj.shape[0]
    theta_grid = np.linspace(0.0, 2 * np.pi, n_grid, endpoint=False)
    idx_grid = fine_idx_grid(fwd, theta_grid)

    iph = np.full((len(kept), n_grid), np.nan)
    vph = np.full((len(kept), n_grid), np.nan)
    for k, ti in enumerate(idx_grid):
        i_n = curr_dq_traj[k]
        if not np.all(np.isfinite(i_n)):
            continue
        iph[:, k] = fwd.phase_map_at_theta(int(ti)) @ i_n
        gU, _, bV = fwd.volt_map_at_theta(int(ti), omega, i_n)
        vph[:, k] = (gU @ i_n + bV)[kept]
    return np.rad2deg(theta_grid), iph, vph


def classify_status(sol):
    if sol is None:
        return "TIMEOUT_OR_ERROR"
    if sol.success:
        return "FEASIBLE"
    if isinstance(sol.diagnostics, dict) and sol.diagnostics.get("fail_reason") == "R0 failed at some nodes":
        return "R0_INFEASIBLE"
    return "STUCK"


def save_point_plot(fwd, omega, torque, rpm, curr_max_used, sol, save_path):
    """Same 9-panel diagnostic layout as the map notebook / feasibility_sweep_infeasible_healthy.ipynb.
    Called unconditionally (even on failure/timeout), as long as there's a finite current to show."""
    status = classify_status(sol)
    volt_max = fwd.drive.volt_max

    sched = sol.curr_dq if sol is not None else None
    if sched is None or not np.any(np.isfinite(sched)):
        return status

    theta, iph, vph = waveforms_for_plot(fwd, omega, sched)

    n_curr = count_peaks_at_limit(np.concatenate([iph, iph[:, :1]], axis=1), curr_max_used) if np.any(np.isfinite(iph)) else 0
    n_volt = count_peaks_at_limit(np.concatenate([vph, vph[:, :1]], axis=1), volt_max) if np.any(np.isfinite(vph)) else 0
    seg_text = segment_label(n_curr, n_volt)

    kept = list(fwd.fault._kept)
    letters = [PHASE_LETTERS[p] for p in kept]

    i_alpha, i_beta, i_x, i_y = alpha_beta_xy(np.deg2rad(theta), sched)
    norm_pw = np.sum(sched ** 2, axis=1)
    cu_pw = np.nansum(iph ** 2, axis=0)
    torque_pw = np.array([
        fwd.drive.torque(omega, sched[k]) if np.all(np.isfinite(sched[k])) else np.nan
        for k in range(sched.shape[0])
    ])

    have_fft = np.all(np.isfinite(iph))
    if have_fft:
        iph_fft = np.abs(np.fft.rfft(iph, axis=1)) / iph.shape[1]
        harmonics = np.arange(iph_fft.shape[1])

    status_color = {"FEASIBLE": "tab:green", "STUCK": "tab:orange",
                     "R0_INFEASIBLE": "tab:red", "TIMEOUT_OR_ERROR": "tab:red"}[status]

    fig, ax = plt.subplots(3, 3, figsize=(18, 13))
    fig.suptitle(
        f"[{status}] {seg_text}   T*={torque:.3f} Nm,  {rpm:.1f} RPM,  curr_max={curr_max_used:.2f} A",
        fontsize=13, color=status_color,
    )

    dq_lbl = [r"$i_{sd}^1$", r"$i_{sq}^1$", r"$i_{sd}^3$", r"$i_{sq}^3$"]
    dq_col = ["tab:blue", "tab:orange", "tab:green", "tab:red"]
    a = ax[0, 0]
    for j in range(fwd.drive.dim):
        a.plot(theta, sched[:, j], color=dq_col[j], label=dq_lbl[j])
    a.set_title("DQ setpoints"); a.set_xlabel(r"$\theta$ [deg]"); a.set_ylabel("DQ current [A]"); a.legend(fontsize=8)

    a = ax[0, 1]
    for p, lt in enumerate(letters):
        a.plot(theta, iph[p], label=f"$i_{{{lt}}}$")
    a.axhline(curr_max_used, color="gray", ls=":"); a.axhline(-curr_max_used, color="gray", ls=":")
    a.set_title("Phase currents"); a.set_xlabel(r"$\theta$ [deg]"); a.set_ylabel("phase current [A]"); a.legend(fontsize=8)

    a = ax[0, 2]
    for p, lt in enumerate(letters):
        a.plot(theta, vph[p], label=f"$v_{{{lt}}}$")
    a.axhline(volt_max, color="gray", ls=":"); a.axhline(-volt_max, color="gray", ls=":")
    a.set_title("Phase voltages"); a.set_xlabel(r"$\theta$ [deg]"); a.set_ylabel("phase voltage [V]"); a.legend(fontsize=8)

    a = ax[1, 0]
    a.plot(theta, cu_pw, color="tab:orange", label=f"mean={np.nanmean(cu_pw):.1f}")
    a.set_title(r"Copper-loss density $\sum_k i_k^2(\theta)$")
    a.set_xlabel(r"$\theta$ [deg]"); a.set_ylabel(r"$\sum_k i_k^2$ [A$^2$]"); a.legend(fontsize=8)

    a = ax[1, 1]
    a.plot(theta, norm_pw, color="tab:orange", label=r"$\|i_s(\theta)\|^2$")
    a.set_title(r"DQ norm $\|i_s\|^2$"); a.set_xlabel(r"$\theta$ [deg]"); a.set_ylabel(r"$\|i_s\|^2$ [A$^2$]"); a.legend(fontsize=8)

    a = ax[1, 2]
    a.plot(i_alpha, i_beta, color="tab:purple")
    a.set_title(r"$\alpha\beta$ current"); a.set_xlabel(r"$i_\alpha$ [A]"); a.set_ylabel(r"$i_\beta$ [A]"); a.set_aspect("equal")

    a = ax[2, 0]
    a.plot(i_x, i_y, color="tab:brown")
    a.set_title("xy current"); a.set_xlabel(r"$i_x$ [A]"); a.set_ylabel(r"$i_y$ [A]"); a.set_aspect("equal")

    a = ax[2, 1]
    a.plot(theta, torque_pw, color="tab:red", label=f"mean={np.nanmean(torque_pw):.3f}")
    a.axhline(torque, color="gray", ls=":", label=f"T*={torque:.3f}")
    a.set_title("Achieved torque (per-node constraint check)")
    a.set_xlabel(r"$\theta$ [deg]"); a.set_ylabel("torque [Nm]"); a.legend(fontsize=8)

    a = ax[2, 2]
    if have_fft:
        width = 0.8 / len(letters)
        for p, lt in enumerate(letters):
            a.bar(harmonics + p * width, iph_fft[p], width=width, label=f"$i_{{{lt}}}$")
        a.set_xlim(-0.5, min(15, harmonics[-1]) + 0.5)
        a.set_title("Phase current spectrum (electrical harmonics)")
        a.set_xlabel("harmonic order"); a.set_ylabel("amplitude [A]"); a.legend(fontsize=8)
    else:
        a.axis("off")
        a.text(0.5, 0.5, "spectrum skipped:\nsome nodes infeasible/NaN", ha="center", va="center", transform=a.transAxes)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(save_path, dpi=110)
    plt.close(fig)
    return status

def probe_t_max(opt, omega, guess=None, start_margin=1.0):
    sol, err = call_with_timeout(PROBE_TIMEOUT_SECONDS, opt.maximize_torque, omega, guess, start_margin)
    if sol is None:
        return None, None, err or "timeout", start_margin
    if not sol.success:
        return None, None, "no_converge", start_margin
    return sol.torque, sol.curr_dq, "ok", sol.diagnostics.get("margin", start_margin)

In [ ]:
results = []
prev_speed_guess = None
prev_margin = 1.0

for rpm in SPEEDS_RPM:
    omega = rpm / const_mech_speed

    drive.curr_max = CURR_MAX
    t_max, curr_at_max, probe_status, prev_margin = probe_t_max(optimizer, omega, guess=prev_speed_guess, start_margin=prev_margin)
    if t_max is None:
        print(f"speed {rpm:.0f} RPM: T_max probe failed ({probe_status}) -- skipping this speed entirely")
        continue
    prev_speed_guess = np.nanmean(curr_at_max, axis=0)
    torq_target = t_max * TORQUE_BACKOFF
    print(f"speed {rpm:.0f} RPM: T_max={t_max:.4f} Nm, fixed target={torq_target:.4f} Nm ({TORQUE_BACKOFF*100:.0f}%) at full curr_max={CURR_MAX} A")

    guess = np.nanmean(curr_at_max, axis=0)
    for frac in REDUCTION_FRACS:
        curr_max_level = CURR_MAX * (1.0 - frac)
        drive.curr_max = curr_max_level

        t0 = time.perf_counter()
        sol, err = call_with_timeout(CELL_TIMEOUT_SECONDS, optimizer.minimize_current, torq_target, omega, guess)
        dt = time.perf_counter() - t0

        fname = f"RPM{rpm:07.1f}_reduce{int(round(frac * 100)):02d}pct_fixedtarget.png"
        fpath = os.path.join(figs_dir, fname)
        try:
            status = save_point_plot(fwd, omega, torq_target, rpm, curr_max_level, sol, fpath)
        except Exception as exc:
            status = f"PLOT_FAILED: {exc!r}"
        if sol is not None and sol.success:
            guess = np.nanmean(sol.curr_dq, axis=0)

        volt_peak = float(np.nanmax(np.abs(waveforms_for_plot(fwd, omega, sol.curr_dq)[2]))) if (
            sol is not None and np.any(np.isfinite(sol.curr_dq))
        ) else float("nan")

        row = {"rpm": rpm, "reduction_pct": frac * 100, "curr_max": curr_max_level,
               "fixed_torque_target": torq_target, "fixed_target_status": status, "volt_peak": volt_peak,
               "seconds": dt}
        results.append(row)
        print(f"    reduce={frac*100:4.0f}%  curr_max={curr_max_level:5.2f}A  "
              f"fixed_target={status:14s} volt_peak={volt_peak:6.2f}V  ({dt:.1f}s)")

drive.curr_max = CURR_MAX

with open(os.path.join(data_dir, "curr_max_sensitivity_neural_flux_torque.pkl"), "wb") as f:
    pickle.dump(results, f)
print(f"\nsaved {len(results)} rows")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
for rpm in SPEEDS_RPM:
    rows = [r for r in results if r["rpm"] == rpm]
    if not rows:
        continue
    fracs = [r["reduction_pct"] for r in rows]
    volts = [r["volt_peak"] for r in rows]
    ax.plot(fracs, volts, marker="o", label=f"{rpm:.0f} RPM")

ax.axhline(VOLT_MAX, color="gray", ls="--", label=f"volt_max={VOLT_MAX} V")
ax.set_xlabel("curr_max reduction [%]")
ax.set_ylabel("peak phase voltage required [V]")
ax.set_title("Current-margin sensitivity (dynamic-old / ActiveSetOptimizer, fixed torque target per speed)")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"{'RPM':>8} {'reduce%':>8} {'status':>16} {'volt_peak':>10}")
for r in results:
    print(f"{r['rpm']:>8.0f} {r['reduction_pct']:>8.0f} {r['fixed_target_status']:>16} {r['volt_peak']:>10.2f}")